In [1]:
import numpy as np

import pyscf
from pyscf.dft.numint import _dot_ao_dm, _contract_rho

from cc2cc import extend
from cc2cc.utils import gen_basis, Grid, rotate

# acetylene_cc-pVDZ_0-1_1_-0.4000_1_18_18
# acetylene_cc-pVDZ_0-1_1_-0.4000_3_27_114

BASIS = "cc-pVDZ"
LEVEL, PERIOD = 1, 2
INDEX_ = (3, 27, 114)
molecule, name = extend("ethylene", "0-1", 1, -0.5, BASIS)
rotate(molecule)

mol = pyscf.M(atom=molecule, basis=gen_basis(molecule, BASIS, True), spin=0)
print(f"Generate data for {name}")

mf = pyscf.scf.RHF(mol)
mf.kernel()
mycc = pyscf.cc.CCSD(mf)
mycc.kernel()
dm1_cc = mycc.make_rdm1(ao_repr=True)
e_cc = mycc.e_tot

mdft = pyscf.scf.RKS(mol)
mdft.xc = "b3lyp"
mdft.kernel()

grids = Grid(mol, level=LEVEL, period=PERIOD)
shls_slice = (0, mol.nbas)
ao_loc = mol.ao_loc_nr()

LEVEL: 1
PERIOD: 2
MAIN_PATH: /home/chenzihao/workspace/cc2cc
DATA_PATH: /home/chenzihao/workspace/cc2cc/data/grids_dft
DATA_CC_PATH: /home/chenzihao/workspace/cc2cc/data/grids_dft
DATA_SAVE_PATH: /home/chenzihao/workspace/cc2cc/data/grids_dft/saved_data
DATA_TEST_PATH: /home/chenzihao/workspace/cc2cc/data/test
STRUCTURE: cnn3d
TEST: False
CUBE_USE: 5
Generate ethylene_-0.5000
Extend 0-1 1 -0.5000
original mol [['C', -0.6672, 0, 0], ['C', 0.6672, 0, 0], ['H', -1.2213, -0.929, 0.0708], ['H', -1.2212, 0.929, -0.0708], ['H', 1.2213, 0.929, -0.0708], ['H', 1.2213, -0.929, 0.0708]]
extend mol [['C', -0.6672, 0, 0], ['C', 0.16720000000000002, 0.0, 0.0], ['H', -1.2213, -0.929, 0.0708], ['H', -1.2212, 0.929, -0.0708], ['H', 1.2213, 0.929, -0.0708], ['H', 1.2213, -0.929, 0.0708]]
Generate data for ethylene_cc-pVDZ_0-1_1_-0.5000
converged SCF energy = -77.1163204056414


<class 'pyscf.cc.ccsd.CCSD'> does not have attributes  converged


E(CCSD) = -77.41661436696731  E_corr = -0.3002939613258541


/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/pyscf/dft/libxc.py:507: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, corresponding to the original definition by Stephens et al. (issue 1480) and the same as the B3LYP functional in Gaussian. To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '


converged SCF energy = -77.6997452127422
n_rad: 40, n_ang: 194


In [2]:
import opt_einsum as oe

dm2_cc = mycc.make_rdm2(ao_repr=True)
expr_rinv_dm2_r = oe.contract_expression(
    "ijkl,i,j,kl->",
    0.5 * dm2_cc
    - 0.5 * oe.contract("pq,rs->pqrs", dm1_cc, dm1_cc)
    + 0.05 * oe.contract("pr,qs->pqrs", dm1_cc, dm1_cc),
    (mol.nao,),
    (mol.nao,),
    (mol.nao, mol.nao),
    constants=[0],
    optimize="optimal",
)

x_mat = grids.vector_to_matrix(grids.coords[:, 0])
y_mat = grids.vector_to_matrix(grids.coords[:, 1])
z_mat = grids.vector_to_matrix(grids.coords[:, 2])

for i, coord in enumerate([[x_mat[INDEX_], y_mat[INDEX_], z_mat[INDEX_]]]):
    ao_i = pyscf.dft.numint.eval_ao(mol, [coord], deriv=2)
    rho_cc = pyscf.dft.numint.eval_rho(mol, ao_i, dm1_cc, xctype="GGA")
    ao_0_i = ao_i[0, 0]
    exc_over_dm_cc_i = -pyscf.dft.libxc.eval_xc("b3lyp", rho_cc)[0]
    exc_over_dm_b3lyp_i = exc_over_dm_cc_i.copy()

    with mol.with_rinv_origin(coord):
        rinv = mol.intor("int1e_rinv")
        print(
            627.509
            * (
                exc_over_dm_cc_i
                + expr_rinv_dm2_r(
                    ao_0_i,
                    ao_0_i,
                    rinv,
                    backend="torch",
                )
                / rho_cc[0]
            )
        )

    rho_cc_1 = np.zeros((3))
    rho_cc_2 = np.zeros((3, 3))

    # # Hessian matrix
    # assert (
    #     np.linalg.norm(dm1_cc.conj().T - dm1_cc) < 1e-10
    # ), "Density matrix is not symmetric."
    # c0 = _dot_ao_dm(mol, ao_i[0][0], dm1_cc, None, shls_slice, ao_loc)
    # rho_cc_1[0] = _contract_rho(ao_i[0][1], c0)
    # rho_cc_1[1] = _contract_rho(ao_i[0][2], c0)
    # rho_cc_1[2] = _contract_rho(ao_i[0][3], c0)
    # rho_cc_2[0, 0] = _contract_rho(ao_i[0][4], c0)
    # rho_cc_2[0, 1] = _contract_rho(ao_i[0][5], c0)
    # rho_cc_2[0, 2] = _contract_rho(ao_i[0][6], c0)
    # rho_cc_2[1, 1] = _contract_rho(ao_i[0][7], c0)
    # rho_cc_2[1, 2] = _contract_rho(ao_i[0][8], c0)
    # rho_cc_2[2, 2] = _contract_rho(ao_i[0][9], c0)
    # rho_cc_2[1, 0] = rho_cc_2[0, 1]
    # rho_cc_2[2, 0] = rho_cc_2[0, 2]
    # rho_cc_2[2, 1] = rho_cc_2[1, 2]

[-11.43040693]


In [7]:
# methane_cc-pVDZ_0-1_1_-0.5000_3_36_29
data1 = np.load("../data/grids_dft_backup/data_methane_cc-pVDZ_0-1_1_-0.5000_1_2.npz")
INDEX_1 = (3, 36, 29)

# methane_cc-pVDZ_0-1_1_-0.3000_3_36_29
data2 = np.load("../data/grids_dft_backup/data_methane_cc-pVDZ_0-1_1_-0.3000_1_2.npz")
INDEX_2 = (3, 36, 29)

print(
    data1["exc_over_dm_cc_grids"][grids.index_2d[INDEX_1]]
    # * data1["rho_inv_4_norm"][0][grids.index_2d[INDEX_1]]
    # * data1["weights"][grids.index_2d[INDEX_1]]
    * 627.509
)
print(
    data2["exc_over_dm_cc_grids"][grids.index_2d[INDEX_2]]
    # * data1["rho_inv_4_norm"][0][grids.index_2d[INDEX_1]]
    # * data1["weights"][grids.index_2d[INDEX_1]]
    * 627.509
)

input1 = data1["rho_cube"][grids.index_2d[INDEX_1]]
input2 = data2["rho_cube"][grids.index_2d[INDEX_2]]
input1[1, :, :, :] = input1[1, :, :, :] ** (1 / 2)
input2[1, :, :, :] = input2[1, :, :, :] ** (1 / 2)

print(np.linalg.norm(input1 - input2))

28.07581309866688
0.0034743830833400937
2.3819808394068862e-15


In [2]:
import numpy as np

data1 = np.load("../data/grids_mrks_backup/data_ethane_cc-pVDZ_0-1_1_0.5000_1_2.npz")
INDEX_1 = (3, 36, 29)

data1.files
# data1["e_cc"]

['e_cc',
 'dm_cc',
 'dm_inv',
 'weights',
 'vxc',
 'exc_real',
 'exc',
 'rho_inv_4_norm',
 'exc1_tr',
 'vxc1_lda',
 'exc1_tr_lda',
 'coords',
 'coords_x_matrix',
 'coords_y_matrix',
 'coords_z_matrix',
 'mol_atom']

In [2]:
-314.3687189736397 / -116.33415559560866, -181.52315758333367 / -77.22352615271255

(2.7022908050016086, 2.350619903374582)

In [35]:
INDEX_1

(0, 10, 161)

In [28]:
import re

with open("/home/chenzihao/workspace/cc2cc_test5/log/train-2310541--1.log") as f:
    lines = f.readlines()
    for line in lines:
        if line.startswith("No file:"):
            match = re.search(r"No file:\s+(.*?)_cc-pVDZ", line)
            if match:
                print(match.group(1), end=" ")

G2RC-1 BH76-H2 FH51-Cl2 G21EA-EA_14n G21IP-IP_79 MRADC-cs-stretch GAPS-CaSe G2RC-24 G21IP-IP_78 GAPS-CaS G2RC-23 SIE4x4-he2+_1.5 SIE4x4-he2+_1.25 RG18-ar2 FH51-HCl HEAVY28-hbr SIE4x4-h2+_1.75 SIE4x4-h2+_1.0 NBPRC-H2 BH76-clf GW100-1333-74-0 GAPS-ScN SIE4x4-h2+_1.25 SIE4x4-he2+_1.75 GAPS-GaAs G21EA-EA_25n G21IP-IP_70 IDISP-h2 GAPS-GaP PA26-h2 MB16_43-H2 HEAVY28-hcl RG18-ne2 HEAVYSB11-br2 G2RC-47 G21IP-IP_80 FH51-H2 NBPRC-h2 G2RC-45 G21EA-EA_14 G2RC-22 G21EA-EA_25 HEAVYSB11-seh RG18-kr2 HEAVYSB11-cl2 G21EA-EA_23n SIE4x4-h2+_1.5 SIE4x4-he2+_1.0 G2RC-51 BH76-hcl GW100-124-38-9 G2RC-40 GAPS-LiZnP S66-16B BH9-06_1R2 S66-18A GAPS-LiZnAs S66-59B PA26-h2p S66-03A BH76-RKT01 S66-02A S66-12B G21EA-EA_15n S66-08B S66-01B S66-04A GAPS-CaF2 S66-54B BH9-09_13R1 GW100-75-15-0 BH9-03_8P2 BH76-hclhts BH76-RKT06 BH76-RKT17 S66-01A GAPS-MgCl2 G21EA-EA_15 AL2X6-alh3 S66-45A NBPRC-BH3 S66-50B MB16_43-AlH3 G2RC-57 GW100-13283-31-3 AL2X6-alcl3 RSE43-P1 GAPS-P G2RC-58 GAPS-CuGaO2 S66-65B BH76-ch3 NBPRC-BF3 HEA

In [26]:
mol1 = "G2RC-1 BH76-H2 FH51-Cl2 G21EA-EA_14n G21IP-IP_79 MRADC-cs-stretch GAPS-CaSe G2RC-24 G21IP-IP_78 GAPS-CaS G2RC-23 SIE4x4-he2+_1.5 SIE4x4-he2+_1.25 RG18-ar2 FH51-HCl HEAVY28-hbr SIE4x4-h2+_1.75 SIE4x4-h2+_1.0 NBPRC-H2 BH76-clf GW100-1333-74-0 GAPS-ScN SIE4x4-h2+_1.25 SIE4x4-he2+_1.75 GAPS-GaAs G21EA-EA_25n G21IP-IP_70 IDISP-h2 GAPS-GaP PA26-h2 MB16_43-H2 HEAVY28-hcl RG18-ne2 HEAVYSB11-br2 G2RC-47 G21IP-IP_80 FH51-H2 NBPRC-h2 G2RC-45 G21EA-EA_14 G2RC-22 G21EA-EA_25 HEAVYSB11-seh RG18-kr2 HEAVYSB11-cl2 G21EA-EA_23n SIE4x4-h2+_1.5 SIE4x4-he2+_1.0 G2RC-51 BH76-hcl GW100-124-38-9 G2RC-40 GAPS-LiZnP S66-16B BH9-06_1R2 S66-18A GAPS-LiZnAs S66-59B PA26-h2p S66-03A BH76-RKT01 S66-02A S66-12B G21EA-EA_15n S66-08B S66-01B S66-04A GAPS-CaF2 S66-54B BH9-09_13R1 GW100-75-15-0 BH9-03_8P2 BH76-hclhts BH76-RKT06 BH76-RKT17 S66-01A GAPS-MgCl2 G21EA-EA_15 AL2X6-alh3 S66-45A NBPRC-BH3 S66-50B MB16_43-AlH3 G2RC-57 GW100-13283-31-3 AL2X6-alcl3 RSE43-P1 GAPS-P G2RC-58 GAPS-CuGaO2 S66-65B BH76-ch3 NBPRC-BF3 HEAVYSB11-geh3 S66-51A FH51-COCl2 AL2X6-alf3 GAPS-CuAlO2 G21EA-EA_10n GAPS-ZnS G2RC-60 S66-32B G21EA-EA_16 S66-51B NBPRC-BCl3 NBPRC-bh3 GAPS-LiCoO2 S66-59A MB16_43-BH3 GAPS-CuScO2 GW100-7784-18-1 G21EA-EA_16n HEAVYSB11-h2se2 RC21-me G2RC-59 S66-60A BH76-RKT15 G2RC-97 G2RC-8 BSR36-ch4 RSE43-E1 G21IP-IP_64 S22-08a BH76-ch3cl G2RC-18 BH76-CH4 S22-10b MB16_43-CH4 S66-09B BH76-clch3clcomp BH76-fch3clcomp2 S66-33B BH76-hch3clts S66-06A BH76-ch3fclts BH76-fch3clcomp1 S66-02B S66-64B MRADC-c2h4-stretch S66-07A S66-31B S66-19A S66-08A S66-01 BH76-fch3clts S66-05A BH76-RKT08 S66-55B BH9-04_22P2 S66-30B S66-05B S66-13B BH9-04_23P2 BH76-clch3clts S30L-29B S66-44A S66-59 BH9-04_50P2 S66-06B G2RC-128 BH9-09_6R1 S66-11A ISO34-P1 RSE43-P8 BH9-08_1P2 S66-12A S66-56B S66-14B S66-09A S66-66A S66-10B S66-03B S66-10A S66-20A S66-51 RSE43-E8 S66-52B S66-60B HEAVYSB11-ge2h6 AL2X6-al2h6 S66-63B G2RC-52 S66-22A S66-20B S66-61B PA26-si2h6 S66-02 S66-23A S66-21A S66-08 S66-62B S66-21B HEAVYSB11-asme2 S66-53B AL2X6-alme2 S66-12 S66-03 S66-05 AL2X6-alme3 S66-06 S66-09 S66-10 S30L-28B IL16-229B GW100-7783-63-3 ALK8-li4_c G2RC-62 G2RC-61 G2RC-66 IL16-152B GW100-558-13-4 IL16-214B GAPS-CaMg2N2 G2RC-67 GW100-75-73-0 GW100-56-23-5 GW100-7783-60-0 BH9-08_9R2 RSE43-P5 RSE43-E5 IL16-230B RSE43-E7 RSE43-P42 PA26-gly RSE43-P31 PA26-phosphapyrrol BHDIV10-ts6 PA26-glyp CDIE20-R21 BHPERI-03r FH51-C2H5CO2H CDIE20-R20 GW100-542-92-7 PA26-phosphapyrrolp RSE43-E6 CDIE20-P20 BHDIV10-ed6 S66-46A ISO34-E30 S66-16A RSE43-E16 FH51-C3H7CN S66-04B S66-15B FH51-C2H5CONH2 S66-64A S66-15A S66-11B S66-13A S66-57B ISO34-P30 S66-14A S66-07B FH51-2-pentyne ISO34-E8 FH51-dimethyloxirane ISO34-E9 FH51-pentadiene FH51-1-pentyne ISO34-P9 ISO34-E29 ISO34-P29 ISO34-P8 BH9-06_31R1 WATER27-OHmH2O4cs ISO34-P18 ISO34-E18 WATER27-OHmH2O4c4 S66-38B FH51-1-pentene FH51-cis-2-pentene S66-37A S66-38A FH51-trans-2-pentene FH51-S_C2H5_2 IL16-212 WATER27-H2O5 GW100-60-29-7 S66-39B S66-42B FH51-diethylamine FH51-C4H9NH2 RSE43-P45 S66-62A ICONF-SI5H12_2 S30L-27B S66-34A GW100-14868-53-2 S66-34B S66-44B ACONF-P_TT S66-46B S66-41B S66-45B ACONF-P_TG ICONF-SI5H12_3 S66-37B ADIM6-AM5 ICONF-SI5H12_1 IL16-202A S66-43B RSE43-E45 ACONF-P_GG S66-35B ICONF-SI5H12_4 S66-40B S66-35A ACONF-P_GX S66-36B ISO34-E10 S66-36A ISO34-P10 S66-61A"
mol1_list = mol1.split(" ")

mol2 = "IDISP-h2 SIE4x4-he2+_1.75 G2RC-23 G21EA-EA_14 G21EA-EA_14n G21IP-IP_79 BH76-clf SIE4x4-he2+_1.25 SIE4x4-h2+_1.5 GAPS-CaS NBPRC-H2 G2RC-45 G21IP-IP_80 RG18-kr2 GAPS-GaAs G21IP-IP_78 HEAVYSB11-br2 G21EA-EA_23n G2RC-47 MRADC-cs-stretch PA26-h2 G21EA-EA_25n GAPS-CaSe G2RC-51 G21IP-IP_70 HEAVY28-hbr HEAVYSB11-seh FH51-H2 G2RC-22 GAPS-GaP G2RC-24 GW100-1333-74-0 FH51-HCl RG18-ar2 G2RC-1 GAPS-ScN SIE4x4-h2+_1.75 HEAVYSB11-cl2 BH76-hcl SIE4x4-h2+_1.25 FH51-Cl2 RG18-ne2 G21EA-EA_25 NBPRC-h2 BH76-H2 HEAVY28-hcl MB16_43-H2 SIE4x4-h2+_1.0 SIE4x4-he2+_1.0 SIE4x4-he2+_1.5 BH76-hclhts BH76-RKT17 GW100-124-38-9 S66-02A S66-59B G2RC-40 S66-16B BH76-RKT06 GAPS-MgCl2 S66-01A GAPS-LiZnP S66-18A S66-01B GAPS-CaF2 G21EA-EA_15 S66-03A GW100-75-15-0 G21EA-EA_15n PA26-h2p S66-04A BH9-03_8P2 GAPS-LiZnAs BH76-RKT01 BH9-06_1R2 S66-54B S66-12B BH9-09_13R1 S66-08B MB16_43-BH3 G21EA-EA_16 G21EA-EA_10n S66-60A GAPS-LiCoO2 HEAVYSB11-h2se2 G21EA-EA_16n GAPS-CuScO2 GW100-13283-31-3 S66-59A GAPS-CuGaO2 G2RC-60 GAPS-ZnS RSE43-P1 NBPRC-BF3 HEAVYSB11-geh3 GAPS-P FH51-COCl2 S66-50B NBPRC-BH3 S66-65B G2RC-59 S66-45A G2RC-58 S66-51A BH76-ch3 GW100-7784-18-1 AL2X6-alf3 RC21-me G2RC-57 NBPRC-BCl3 S66-32B AL2X6-alcl3 GAPS-CuAlO2 S66-51B MB16_43-AlH3 NBPRC-bh3 AL2X6-alh3 S22-08a G2RC-8 S22-10b MB16_43-CH4 G21IP-IP_64 G2RC-97 G2RC-18 BSR36-ch4 BH76-RKT15 BH76-ch3cl RSE43-E1 BH76-CH4 S66-13B MRADC-c2h4-stretch BH9-04_23P2 S66-01 S66-44A BH76-fch3clcomp1 BH76-clch3clts S30L-29B S66-19A S66-05B S66-05A BH76-clch3clcomp BH76-fch3clcomp2 BH76-ch3fclts S66-30B BH9-04_22P2 S66-31B S66-06A S66-02B BH76-RKT08 S66-08A S66-09B S66-33B S66-64B S66-55B BH76-hch3clts S66-07A BH76-fch3clts BH9-09_6R1 S66-03B BH9-04_50P2 G2RC-128 S66-09A S66-59 S66-10A S66-11A S66-06B S66-14B ISO34-P1 S66-56B S66-12A RSE43-P8 S66-10B S66-66A BH9-08_1P2 S66-61B AL2X6-al2h6 S66-63B RSE43-E8 S66-22A S66-51 G2RC-52 PA26-si2h6 S66-20A S66-52B HEAVYSB11-ge2h6 S66-60B S66-20B S66-23A S66-08 S66-21B S66-21A S66-53B HEAVYSB11-asme2 S66-02 S66-62B AL2X6-alme2"
mol2_list = mol2.split(" ")

In [27]:
print(len(mol1_list))
print(len(mol2_list))

215
195
